# Train/Test Split

Reads `dataset/model_ready/featured.parquet` (the cleaned + feature-engineered, pre-split dataset written by `notebook/data-processing.ipynb`), splits into train/test, exports `dataset/model_ready/{train,test}.parquet`, and QA-checks the split itself (row counts, no dates crossing the cutoff wrong, target NaN rates near the December-2025 boundary).

In [1]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd

from utils.data_preprocessing import prepare_forecast_data

## 1. Load featured dataset

In [2]:
featured = pd.read_parquet(Path(prepare_forecast_data.MODEL_READY_DIR) / "featured.parquet")

print(featured.shape)
featured.head()

(1340034, 62)


,Kode Barang,Nama Cabang,Tanggal,Kuantitas,Kategori Barang,Nama Barang,day_of_week,day_of_month,month,is_weekend,...,kota,has_shopee,has_gofood,has_grabfood,can_order_online,target_lead_time_cumulative,branch_avg_daily_qty,branch_demand_cv,branch_volume_tier,branch_age_days
0,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-01,235.0,Barang Jadi (FG),Ayam Kebuli (0.9),0,1,1,False,...,Kota Tangerang,True,True,True,True,338.0,1670.328571,0.438582,flagship,0
1,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-02,147.0,Barang Jadi (FG),Ayam Kebuli (0.9),1,2,1,False,...,Kota Tangerang,True,True,True,True,191.0,1670.328571,0.438582,flagship,1
2,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-03,85.0,Barang Jadi (FG),Ayam Kebuli (0.9),2,3,1,False,...,Kota Tangerang,True,True,True,True,106.0,1670.328571,0.438582,flagship,2
3,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-04,106.0,Barang Jadi (FG),Ayam Kebuli (0.9),3,4,1,False,...,Kota Tangerang,True,True,True,True,730.0,1670.328571,0.438582,flagship,3
4,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-05,155.0,Barang Jadi (FG),Ayam Kebuli (0.9),4,5,1,False,...,Kota Tangerang,True,True,True,True,575.0,1670.328571,0.438582,flagship,4


## 2. Split & export

In [3]:
train, test = prepare_forecast_data.split_train_test(featured)
prepare_forecast_data.export_splits(train, test)

print(f"train: {len(train)} baris, test: {len(test)} baris")

train: 1291694 baris, test: 48340 baris


## 3. QA — split integrity

In [4]:
assert len(train) + len(test) == len(featured), "train+test tidak sama dengan featured"
assert train["Tanggal"].max() < prepare_forecast_data.TEST_START, "Ada tanggal train >= cutoff"
assert test["Tanggal"].min() >= prepare_forecast_data.TEST_START, "Ada tanggal test < cutoff"

print("Split integrity QA: OK")

Split integrity QA: OK


In [5]:
# NaN rate mendekati boundary Des-2025 diharapkan: target_h{n}/target_lead_time_cumulative
# butuh n/lead_time_days hari ke depan yang datanya sudah tidak tersedia setelah 2025-12-31.
for h in range(1, 8):
    print(f"target_h{h} NaN rate (test): {test[f'target_h{h}'].isna().mean():.4f}")
print(f"target_lead_time_cumulative NaN rate (test): {test['target_lead_time_cumulative'].isna().mean():.4f}")

target_h1 NaN rate (test): 0.0348
target_h2 NaN rate (test): 0.0695
target_h3 NaN rate (test): 0.1042
target_h4 NaN rate (test): 0.1388
target_h5 NaN rate (test): 0.1734
target_h6 NaN rate (test): 0.2079
target_h7 NaN rate (test): 0.2423
target_lead_time_cumulative NaN rate (test): 0.0794
